# 07 — Fiziksel Gecerlilik ve Hata Etki Haritasi (Katman 5 & 6)
Uc durumlu karar sozlesmesini (`ACCEPTED`, `REJECTED`, `NOT_COMPUTABLE`) ve
bagimsiz olcumlerin korunmasi ilkesini somut bir biyomekanik senaryoyla dogrular.

## Mimari Sozlesme ve Arastirma Ilkeleri

Bu notebook, **PhysioLoad AI — Olcum Duyarli Secici Biyomekanik Analiz Mimarisi**'nin bir parcasidir:
1. **Once olc, sonra guvenilirligini sinayarak raporla:** Her biyomekanik cikti bagimsiz degerlendirilir.
2. **Uc durumlu karar sozlesmesi:** `ACCEPTED` (Kabul), `REJECTED` (Ret), `NOT_COMPUTABLE` (Hesaplanamaz).
3. **Ayri dogrulama hatti:** Referans eklem verileri asla model tahminine girdi yapilmaz; yalnizca hata analizi ve A0-A4 karsilastirmasinda kullanilir.
4. **Veri sizintisini onleme:** Kisi ve deneme bazli boluntuleme (ayni katilimci train/val/test arasinda karismaz).
5. **Esik secimi:** Sayisal esikler yalnizca validation bolumunde belirlenir; test kumesi karsilastirma icin saklanir.


In [1]:
import sys
from pathlib import Path

# Proje kok dizini ve ortam kurulumu
here = Path.cwd().resolve()
code_root = next(p for p in (here, *here.parents) if (p / 'physioload_env.py').exists())
sys.path.insert(0, str(code_root))

from physioload_env import setup_environment
env = setup_environment(root=code_root)
print("Ortam Basariyla Yuklendi:")
print(env.summary())


Ortam Basariyla Yuklendi:
context       : vscode local
user          : berk-can
processed ver : v1
device        : cpu
code root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
artifacts root: C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai
data root     : C:\Users\Berk\Desktop\BİTİRME PROJESİ\physioload ai


In [2]:
from physioload.physical_validity import PhysicalValidityContext
from physioload.quality import QualityFeatures, QualityGate, MeasurementStatus

# Senaryo: Sag sagittal goruntu
# - Diz acisi: Yuksek guven, dogru aci -> ACCEPTED
# - Omuz acisi: Dusuk gorunurluk (ortulme var) -> REJECTED
# - Dis moment: Metrik kalibrasyon yok -> NOT_COMPUTABLE

gate = QualityGate()

# 1. Diz Acisi Degerlendirmesi
knee_feat = QualityFeatures(visibility=0.92, pose_confidence=0.94, view_deviation_deg=4.0, geometry_penalty=0.04, scale_valid=False)
knee_dec = gate.evaluate(knee_feat, requested_outputs=['joint_angle'])

# 2. Omuz Acisi Degerlendirmesi (Ortulme / Dusuk Gorunurluk)
shoulder_feat = QualityFeatures(visibility=0.45, pose_confidence=0.50, view_deviation_deg=4.0, geometry_penalty=0.08, scale_valid=False)
shoulder_dec = gate.evaluate(shoulder_feat, requested_outputs=['joint_angle'])

# 3. Dis Moment Degerlendirmesi (Olceksiz)
moment_dec = gate.evaluate(knee_feat, requested_outputs=['external_moment'], physical_context=PhysicalValidityContext(metric_scale_valid=False))

print("=== COKLU OLCUM SECICI KARAR RAPORU ===")
print(f"Diz Acisi   : {knee_dec.outputs['joint_angle'].status.value} (Skor: {knee_dec.outputs['joint_angle'].score:.2f})")
print(f"Omuz Acisi  : {shoulder_dec.outputs['joint_angle'].status.value} (Nedenler: {shoulder_dec.outputs['joint_angle'].reasons})")
print(f"Dis Moment  : {moment_dec.outputs['external_moment'].status.value} (Nedenler: {moment_dec.outputs['external_moment'].reasons})")

# Kritik Dogrulama
assert knee_dec.outputs['joint_angle'].status == MeasurementStatus.ACCEPTED
assert shoulder_dec.outputs['joint_angle'].status == MeasurementStatus.REJECTED
assert moment_dec.outputs['external_moment'].status == MeasurementStatus.NOT_COMPUTABLE
print("\nOmuz acisi reddedilirken ve dis moment hesaplanamazken, diz acisinin basariyla KABUL edildigi dogrulandi.")


=== COKLU OLCUM SECICI KARAR RAPORU ===
Diz Acisi   : ACCEPTED (Skor: 0.94)
Omuz Acisi  : REJECTED (Nedenler: ('low_visibility', 'low_pose_confidence'))
Dis Moment  : NOT_COMPUTABLE (Nedenler: ('metric_scale_not_validated', 'load_magnitude_missing', 'load_application_point_missing', 'load_direction_missing'))

Omuz acisi reddedilirken ve dis moment hesaplanamazken, diz acisinin basariyla KABUL edildigi dogrulandi.
